# ST 554 Final Project

*By: Cass Crews*

## Introduction


## Module Loading and Data Cleaning

Before we load the data and clean it, we need to load the modules and functions we will need in this notebook. 

In [1]:
# Reading in modules and sub-modules, also initiating spark sequence
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, \
    VectorIndexer, OneHotEncoder, Interaction, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import PCA
from pyspark.sql.types import StructType
from pyspark.sql.functions import col
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 21:44:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/26 21:44:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/26 21:44:05 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/26 21:44:05 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


We are now ready to load the dataset. We'll initially read it in as a `pandas` dataframe before converting to a SQL-style `pyspark` dataframe. Once we've read the file in, we will extract the first few rows to ensure the data loaded correctly. 

In [2]:
# Reading in the data
power_pddf = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")

# Printing the first few rows
power_pddf.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


This looks correct. Let's apply a few validation checks before converting to a `pyspark` dataframe. We'll start by checking for missing values. 

In [3]:
# Checking for missing values
power_pddf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47174 entries, 0 to 47173
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Temperature            47174 non-null  float64
 1   Humidity               47174 non-null  float64
 2   Wind_Speed             47174 non-null  float64
 3   General_Diffuse_Flows  47174 non-null  float64
 4   Diffuse_Flows          47174 non-null  float64
 5   Power_Zone_1           47174 non-null  float64
 6   Power_Zone_2           47174 non-null  float64
 7   Power_Zone_3           47174 non-null  float64
 8   Month                  47174 non-null  int64  
 9   Hour                   47174 non-null  int64  
dtypes: float64(8), int64(2)
memory usage: 3.6 MB


None of the variables are missing, and all are stored as numeric -- as they should be. 

Of course, there could be hidden missing values that have a non-`NaN` missing value code. We can generate some basic summary statistics in an attempt to surface such values. 

In [4]:
# Generating validation summary statistics
power_pddf.describe()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
count,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000
mean,18.813220,68.288398,1.961621,182.531180,74.987211,32335.168690,21027.204976,17831.197608,6.510599,11.488383
std,5.813341,15.560330,2.349351,264.431856,124.256146,7130.013305,5199.787153,6622.590470,3.437367,6.921920
min,3.247000,11.340000,0.050000,0.004000,0.011000,13895.696200,8560.081466,5935.174070,1.000000,0.000000
25%,14.420000,58.320000,0.078000,0.062000,0.122000,26289.581305,16956.121510,13121.927710,4.000000,5.000000
50%,18.780000,69.890000,0.086000,4.829000,4.279500,32261.596960,20804.671935,16406.308170,7.000000,11.000000
75%,22.910000,81.500000,4.915000,318.900000,101.000000,37317.446810,24697.888273,21633.372830,9.000000,17.000000
max,40.010000,94.800000,6.483000,1163.000000,936.000000,52146.859050,37408.860760,47598.326360,12.000000,23.000000


In [5]:
power_pddf.Month.value_counts()

Month
3     4057
7     4029
10    4026
1     4014
8     3999
5     3997
6     3913
9     3913
4     3893
11    3877
12    3868
2     3588
Name: count, dtype: int64

While I am not an expert in any of these metrics, none of the minimum or maximum values clearly indicate missing value codes. The dataset's UCI Machine Learning Repository page indicated no true missing values, but it's always good practice to check for ourselves! 

We are ready to convert to a `pyspark` dataframe. After the conversion, we will again extract the first few rows to confirm they match the first few rows of the `pandas` dataframe. 

In [6]:
# Converting to a pyspark dataframe
power_df = spark.createDataFrame(power_pddf)

# Printing first five rows
power_df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

Everything seems to be in order. Onto building the machine learning model! 

## Fitting the Model

As was mentioned in the introduction, we will tune an elastic net model for the purpose of predicting `Power_Zone_3`. We will construct an `MLlib` pipeline to do so, as this will make it much easier to predict power consumption for streamed values. Thus, we need to construct the appropriate transformers. 

We'll work through these sequentially, starting with the creation of a binary predictor from `Hour`, the hour of the day. First, we need to determine how `Hour` was stored when we converted to the `pyspark` dataframe. 

In [7]:
power_df.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

`Hour` is stored as a `bigint`, so we will need to cast it to a `double` type before we create the binary variable; this binary variable will indicate whether the observation corresponds to the early-morning hours (between midnight and 6am) or not, so we need to be able to pass `Binarizer` a threshold that is not an integer. 

In [8]:
# Creating double version of Hour
hour_conv_step = SQLTransformer(statement = """
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_db FROM __THIS__
    """)

We can use this `double`-cast column to generate the early-morning binary variable. 

In [9]:
# Creating binary early morning indicator
hour_binar_step = Binarizer(threshold = 6.5, inputCol = "Hour_db", outputCol = "early_morn_bin")

Our next transformer should create month dummies from `Month`. We'll use a one-hot encoder transformer for this step. However, 

In [10]:
# Creating month dummies
month_dummy_step = OneHotEncoder(inputCol = "Month", outputCol = "Month_enc")

Note that the output column is a single column although we created dummies for each month. This is because `OneHotEncoder()` expects the dummies to be added to a single features column of downstream, and, therefore, stores an observation's month dummy values in a vector rather than creating multiple unnecessary columns. 

The next step of the pipeline is more complex. We want to distill the atmospheric variables into their principle components. Thus, we will fit a *Principal Components Analysis (PCA)* to `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows`. We will then retain the two principal components that account for the most variability across the atmospheric variables. 

Our first step is to combine the atmospheric variables into a single `features_pca_raw` column to be fed to the `PCA()` transformer. 

In [11]:
# Creating feature column for PCA
pca_assembly_step = VectorAssembler(inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"],
                              outputCol = "features_pca_raw")

We now need to standardize each feature, as different scales can influence a PCA fit. This simply requires us to pass in the `features_pca_raw` column, as `StandardScaler()` will parse the column and standardize each feature. 

In [12]:
# Standardizing features for PCA
pca_scaling_step = StandardScaler(inputCol = "features_pca_raw", outputCol = "features_pca_stand", withMean = True, withStd = True)

We can now apply PCA without worrying about any scaling issues. In specifying `k = 2`, this transformer will return the first two principal components in a single column of vectors; each vector will be of length 2. 

In [13]:
# Applying PCA to scaled atmospheric features
pca_step = PCA(inputCol = "features_pca_stand", outputCol = "princ_comps", k = 2)

As we are constructing a pipeline that leads to an elastic net model, we need to rename `Power_Zone_3` `label` in line with MLlib convention. We'll use this as an opportunity to remove the variables we no longer need. 

In [14]:
# Renaming Power_Zone_3 label and dropping some variables
clean_step = SQLTransformer(statement = """
    SELECT Power_Zone_3 AS label, princ_comps, early_morn_bin, Power_Zone_1, Power_Zone_2, Month_enc
    FROM __THIS__
    """)

We are ready for the final transformation step: to compile the principal components, `early_morn_bin`, `Power_Zone_1`, `Power_Zone_2`, and `Month_enc` into a single `features` column.

In [15]:
# Compiling final feature set into a features column
assembly_step = VectorAssembler(inputCols = ["princ_comps", "early_morn_bin", "Power_Zone_1", "Power_Zone_2", "Month_enc"], 
                                outputCol = "features")

It's time to specify the model for our pipeline, construct the pipeline, and construct a grid of hyperparameters. As the elastic net model involves both an $L_1$ and an $L_2$ penalty, we will naturally need to tune two hyperparameters. When using the `LinearRegression()` estimator to fit an elastic net model, the two hyperparameters are `regParam` and `elasticNetParam`. `regParam` determines the overall regularization amount across both penalties, while `elasticNetParam` determines the relative severity of each penalty; `elasticNetParam = 0` corresponds to a pure ridge regression, and `elasticNetParam = 1` corresponds to a pure LASSO regression. 

For each hyperparameter, we will consider the same range of 11 values: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, and 1. We will evaluate predictive performance for each combination of hyperparameter values, an 11-by-11 grid of 121 combinations. 

An important note: `LinearRegression()` handles feature standardization for us via the default `standardization=True`, meaning we do not need to add another standardization transformation to the pipeline prior to fitting elastic net models. 

In [16]:
# Specifying the estimator
en_model = LinearRegression()

# Constructing the pipeline
pipeline = Pipeline(stages = [hour_conv_step, hour_binar_step, month_dummy_step, pca_assembly_step, pca_scaling_step, pca_step, clean_step, assembly_step, en_model])

# Constructing the 11x11 hyperparameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(en_model.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(en_model.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

We are now ready to construct a `CrossValidator()` object, specifying that we want to use 5-fold cross validations to tune the elastic net model and select the best overall model based on cross validated root mean squared error (RMSE). Once we have constructed the object, we will fit it to identify the best combination of hyperparameters. 

In [17]:
# Constructing CrossValidator object
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(), # rmse by default
                          numFolds = 5, # 5-fold CV
                          seed = 5) # setting seed for reproducibility

# Fitting the object
en_cv = crossval.fit(power_df)

26/04/26 21:44:20 WARN Instrumentation: [d40ba817] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 21:44:21 WARN Instrumentation: [d40ba817] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/26 21:44:23 WARN Instrumentation: [cb370a5f] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 21:44:23 WARN Instrumentation: [cb370a5f] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/26 21:44:23 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/04/26 21:44:23 ERROR LBFGS: Failure again! Giving up and returning. Maybe the objective is just poorly behaved?
26/04/26 21:44:25 WARN Instrumentation: [4343c5d3] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 21:44:25 WARN Instrumentation: [4343c5d3] Cholesky solver failed due to singular covarianc

We have now tuned the elastic net model. Let's extract the cross-validated RMSE values and corresponding hyperparameter values for the ten models with the smallest RMSE. 

In [18]:
# Creating empty list to append model metric-hyperparameter pairs to
tune_list = []

# Appending model metric-hyperparameter pairs
for i in range(len(paramGrid)):
    tune_list.append([en_cv.avgMetrics[i], paramGrid[i].values()])

# Sorting the list by cross-validated log-loss
tune_list.sort(key = lambda x: x[0])

# Printing top 10 models
tune_list[0:10]

[[2174.9952911524388, dict_values([0.1, 0.1])],
 [2174.9954859954587, dict_values([0.05, 0.1])],
 [2174.9955056252993, dict_values([0.1, 0.05])],
 [2174.995536629761, dict_values([0.05, 0.25])],
 [2174.995823030331, dict_values([0.25, 0.05])],
 [2174.9960859644852, dict_values([0.05, 0.05])],
 [2174.996195662562, dict_values([0.1, 0.0])],
 [2174.9962089460064, dict_values([0.05, 0.0])],
 [2174.996217377181, dict_values([0.25, 0.0])],
 [2174.996231477561, dict_values([0.0, 0.5])]]

These results are interesting: the top-performing model has a relatively large amount of regularization (`regParam` = 0.98) with regularization relatively balanced across the two penalty terms (`elasticNetParam` = 0.5). However, the second-best-performing model has very little total regularization (`regParam` = 0.05) and relies mostly on the ridge penalty (`elasticNetParam` = 0.05). This may be an indication that predicting `Power_Zone_3` is relatively robust to regularization scale and form. 

Overall, the cross-validated RMSE for each of the top models is roughly a third of `Power_Zone_3`'s standard deviation (6622.59), so our model is providing substantial benefit relative to simply using the sample mean to predict future power usage. 

To understand the quality of the training set fit, we can calculate the RMSE from predicting on our full dataset. This is as simple as using the `.evaluate()` method for `RegresssionEvaluator()` and passing `en_cv.transform(power_df)`. Our dataset will go through the pipeline and be "transformed" into predictions using the best-performing model fit to the same dataset. 

In [19]:
# Calculating full dataset RMSE
training_rmse = RegressionEvaluator().evaluate(en_cv.transform(power_df))
print(training_rmse)

2174.448953950987


As the model has seen all these observations before, it is not surprising that the training RMSE is a bit smaller than the cross-validated RMSE. 

To better understand all the work we've done up to this point, we can produce a final dataframe of true values, predictions, and residuals for the training set. We'll do this now and print the first few rows of data. 

In [20]:
# Creating table of true power values, predictions, and residuals
true_vs_predictions = en_cv.transform(power_df).select("label", "prediction").withColumn("residual", col("label") - col("prediction"))

# Extracting first five rows
true_vs_predictions.show(5)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20935.289364929424|-694.3255049294239|
|20131.08434|18701.428523018487|1429.6558169815144|
|19668.43373|18237.065138663602|1431.3685913363988|
|18899.27711|17615.794827031168| 1283.482282968831|
|18442.40964|17017.326343175464|1425.0832968245377|
+-----------+------------------+------------------+
only showing top 5 rows


Once the pipeline is set up, the MLlib workflow is truly streamlined. We easily obtained predictions appended to our processed dataframe and could easily create a table of true values and predictions for `Power_Zone_3`. 

## Streaming New Observations and Predicting Power Consumption

Now that we have a final fitted elastic net model, we can stream incoming observations and effectively put our model into production. This could be particularly helpful if we lost the ability to explicitly measure `Power_Zone_3` but still needed to know how much power was being consumed in that area. 

To showcase the ability to deploy our model on streamed data, we will simulate the streaming process by creating a `.py` file that randomly draws observations from a held-out set of observations and makes them visible when we begin streaming. Let's prepare for the stream by specifying the schema for the inbound data and setting up a `readStream`. Fortunately, the schema will match that of `power_df`. The data will come in via `.csv` files with headers and land in the `streamed_obs` folder in our local environment. We'll need to indicate all three facts for the `readStream`. 

In [21]:
# Specifying our streaming data's input schema
myschema = power_df.schema

# Speciying the streaming data itself, which will arrive in streamed_obs folder
# Note that we need to specify that the data have headers
power_data = spark.readStream.schema(myschema) \
    .option("header", "true").csv("streamed_obs")

We are now ready to construct the separate transformations we will apply to the streamed `power_data` and indicate that we want to join these transformed dataframes on the true value of `Power_Zone_3` (denoted `label` after the transformations). 

The first transformation will effectively recreate `true_vs_predictions` above for the streamed observations, producing a true `label` value, `prediction`, and `residual`. The second transformation will simply rename `Power_Zone_3` `label` so we can merge the dataframes on `label`. 

A note on the join: we are performing an inner join, but the join type should not matter as we are merging transformations for the same inbound observations. 

In [22]:
# Predicting Power_Zone_3 and producing residual
predictions = en_cv.transform(power_data).select("label", "prediction").withColumn("residual", col("label") - col("prediction"))

# Renaming Power_Zone_3 label in original streamed data
power_data_rename = power_data.withColumnRenamed("Power_Zone_3", "label")

# Merging transformed dataframes for streamed observations
joined = predictions \
                .join(power_data_rename, "label", "inner") 

Now that we've constructed the three objects we need to modify the streamed data, we simply need to start the stream. We'll indicate we want to append outputs as new data come in, and we'll also indicate we simply want the results printed to the console. 

In [44]:
# Starting the stream to perform the two transformations and join results
# setting truncate to false to ensure we see all the resultant rows
power_prediction_streamed = joined.writeStream.outputMode("append") \
                .format("console") \
                .option("truncate", "false") \
                .start()

26/04/26 22:37:06 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-cd031afe-2511-4bd4-b01e-2b8e876cced2. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/26 22:37:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|24901.75732|25508.63264483885 |-606.8753248388493 |27.73      |27.91   |4.911     |0.077                |0.141        |29354.95017 |20008.86076 |7    |2   |
|14150.32258|13347.852255983911|802.4703240160889  |8.63       |87.3    |0.075     |0.022                |0.148        |23003.23404 |13741.46341 |3    |3   |
|27830.97179|27210.381904835405|620.5898851645943  |31.02      |47.15   |4.905     |819.0                |121.7  

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12214.46809|15375.101411071118|-3160.633321071118 |22.91      |83.2    |0.083     |316.7                |219.2        |38568.05252 |22451.45228 |10   |13  |
|15558.22329|17237.90694993451 |-1679.6836599345097|14.84      |75.5    |0.084     |0.037                |0.167        |38065.39924 |32133.78337 |12   |20  |
|18580.08097|19509.312606410782|-929.2316364107828 |17.47      |84.8    |4.919     |0.055                |0.126  

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|8472.289157|9913.37324147037  |-1441.0840844703707|18.67      |72.0    |0.072     |53.64                |53.53        |25667.69231 |20383.8843  |11   |9   |
|16412.53012|17012.340678794277|-599.8105587942773 |18.3       |28.85   |0.089     |366.7                |432.9        |33484.55696 |21242.55319 |1    |16  |
|16883.56275|19768.723384384033|-2885.160634384032 |20.71      |73.6    |4.922     |817.0                |61.3   

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|13848.3871 |12534.815174649953|1313.5719253500465 |15.63      |59.58   |0.079     |0.07                 |0.152        |22255.65957 |12508.53659 |3    |4   |
|20087.05167|21418.206106077614|-1331.1544360776134|23.89      |59.62   |4.916     |0.102                |0.067        |44762.8884  |30133.19502 |10   |20  |
|16019.20768|18054.878552775604|-2035.670872775605 |14.57      |67.49   |0.087     |0.051                |0.141  

...

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual          |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14691.23596|14615.977731472936|75.2582285270637  |23.83      |64.09   |4.924     |294.1                |225.7        |34706.54867 |19530.56133 |9    |13  |
|38464.26778|36062.506457761665|2401.761322238337 |25.75      |75.9    |4.905     |5.927                |4.846        |47425.91362 |29532.91139 |7    |20  |
|24160.66946|24202.90074729245 |-42.23128729244854|25.33      |66.86   |4.904     |374.8                |150.6       

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9968.787515|11774.535663253366|-1805.7481482533658|17.54      |45.91   |0.075     |258.0                |77.3         |31537.64259 |26330.77631 |12   |16  |
|11827.2    |13736.647935762689|-1909.4479357626878|20.46      |84.5    |0.067     |231.7                |181.8        |25850.06623 |16267.35967 |6    |8   |
|37813.55649|31560.922053550472|6252.634436449531  |32.73      |27.78   |4.924     |134.6                |143.2 

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17447.71084|17731.45939188415 |-283.7485518841495 |8.49       |76.9    |0.083     |0.084                |0.141        |28362.53165 |18320.97264 |1    |0   |
|16755.30364|16759.547928627508|-4.244288627509377 |21.56      |70.2    |4.926     |637.8                |592.3        |34068.98361 |22183.28173 |5    |16  |
|33903.94984|29498.28967343922 |4405.660166560781  |26.52      |85.8    |4.908     |0.095                |0.119 

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|23022.27692|19483.31964770304 |3538.9572722969606 |23.69      |64.49   |0.054     |915.0                |42.94        |35602.64901 |18729.72973 |6    |13  |
|31553.4728 |32107.730175757046|-554.2573757570462 |32.39      |37.95   |4.91      |855.0                |61.56        |43719.86711 |27573.41772 |7    |12  |
|31553.4728 |32107.730175757046|-554.2573757570462 |32.39      |37.95   |4.91      |855.0                |61.56 

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9640.336134|7159.776853127594 |2480.559280872405  |12.64      |74.0    |0.078     |0.07                 |0.145        |21377.94677 |15718.93219 |12   |6   |
|26408.03347|28673.013343784478|-2264.9798737844794|27.35      |71.0    |4.919     |593.0                |109.7        |38004.51827 |24383.5443  |7    |16  |
|13833.25301|12914.821393345068|918.4316166549324  |18.11      |41.89   |0.09      |0.037                |0.148 

In [43]:
power_prediction_streamed.stop()

26/04/26 22:36:43 WARN DAGScheduler: Failed to cancel job group a73cac9d-862a-435c-aaa3-f09f60205b99. Cannot find active jobs for it.
26/04/26 22:36:44 WARN DAGScheduler: Failed to cancel job group a73cac9d-862a-435c-aaa3-f09f60205b99. Cannot find active jobs for it.
